# Data Preprocessing

Prepare the dataset for machine learning by separating features and target, splitting the data, and handling missing values and categorical variables.

In [19]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("../data/raw/student_mental_health.csv")

df.head()

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


### Features and Target

Separate the input features (X) from the target variable (y).

Remove Student_ID because it is only an identifier and does not provide useful predictive information.

In [20]:
X = df.drop(columns=["Student_ID","Mental_Health_Risk"])
y = df["Mental_Health_Risk"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (2000, 15)
Target shape: (2000,)


### Train/Test Split

Split the dataset into 80% training data and 20% testing data.

Use stratification to preserve the target class distribution in both sets.

In [21]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

X_train: (1600, 15)
X_test: (400, 15)

Training distribution:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Testing distribution:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


### Feature Types

Separate numerical and categorical features to apply appropriate preprocessing strategies.

In [22]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()
print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']

Categorical features: ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


### Missing Values

Identify missing values in the training features before applying imputation.

In [23]:
print("Numerical missing values :")
print(X_train[numeric_features].isnull().sum())

print("Categorical missing values :")
print(X_train[categorical_features].isnull().sum())

Numerical missing values :
Age                      0
Year_of_Study            0
Sleep_Hours             54
Study_Hours_Per_Week     0
GPA                     36
Social_Media_Hours      48
Screen_Time_Hours        0
Financial_Stress        76
Family_Support          36
Attendance_Rate         39
dtype: int64
Categorical missing values :
Gender                             0
Physical_Activity                  0
Part_Time_Job                      0
Diet_Quality                      55
Previous_Mental_Health_History     0
dtype: int64


### Missing Value Imputation

Use the median for numerical features and the most frequent value for categorical features.

Fit imputers on training data only to prevent data leakage.

In [24]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")
categoric_imputer = SimpleImputer(strategy="most_frequent")

X_train[numeric_features] = numeric_imputer.fit_transform(
    X_train[numeric_features]
)

X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

X_train[categorical_features] = categoric_imputer.fit_transform(
    X_train[categorical_features]
)

X_test[categorical_features] = categoric_imputer.transform(
    X_test[categorical_features]
)

In [25]:
print("Training missing values:", X_train.isnull().sum())
print("Testing missing values:", X_test.isnull().sum())

Training missing values: Age                               0
Gender                            0
Year_of_Study                     0
Sleep_Hours                       0
Study_Hours_Per_Week              0
GPA                               0
Social_Media_Hours                0
Screen_Time_Hours                 0
Physical_Activity                 0
Financial_Stress                  0
Family_Support                    0
Attendance_Rate                   0
Part_Time_Job                     0
Diet_Quality                      0
Previous_Mental_Health_History    0
dtype: int64
Testing missing values: Age                               0
Gender                            0
Year_of_Study                     0
Sleep_Hours                       0
Study_Hours_Per_Week              0
GPA                               0
Social_Media_Hours                0
Screen_Time_Hours                 0
Physical_Activity                 0
Financial_Stress                  0
Family_Support                    0
At

### Categorical Encoding

Apply One-Hot Encoding to convert categorical features into numerical values.

Fit the encoder on training data only to prevent data leakage.

In [26]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

### Combine Numerical and Encoded Features

Convert the encoded arrays into DataFrames with column names and original indexes. Then combine them with the numerical features using `pd.concat(axis=1)`.

This produces the final training and testing datasets containing only numerical values, ready for machine learning.

In [27]:
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_columns,
    index=X_test.index
)

X_train = pd.concat(
    [X_train[numeric_features] , X_train_encoded], axis=1
)

X_test = pd.concat(
    [X_test[numeric_features] , X_test_encoded ], axis=1
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

X_train.head()

Training shape: (1600, 23)
Testing shape: (400, 23)


,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate,...,Physical_Activity_High,Physical_Activity_Low,Physical_Activity_Moderate,Part_Time_Job_No,Part_Time_Job_Yes,Diet_Quality_Average,Diet_Quality_Good,Diet_Quality_Poor,Previous_Mental_Health_History_No,Previous_Mental_Health_History_Yes
726,21.0,2.0,5.2,18.6,2.59,6.2,10.4,7.3,7.7,76.1,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1384,23.0,1.0,4.9,12.3,2.23,6.4,8.9,4.4,7.2,81.7,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1579,19.0,2.0,7.2,17.0,2.21,5.7,7.2,5.4,7.9,90.6,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1098,23.0,1.0,8.3,19.5,2.87,6.9,10.4,7.5,4.8,83.5,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
582,22.0,3.0,7.2,23.8,2.47,5.6,8.0,7.2,7.8,74.6,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0


## Check Missing Values in Train & Test

In [28]:
print("Train count missing values" , X_train.isnull().sum().sum())
print("Test count missing values" , X_train.isnull().sum().sum())


Train count missing values 0
Test count missing values 0


### Feature Scaling

Feature scaling puts numerical features on comparable scales.

We use StandardScaler to transform numerical features to a mean of approximately 0 and a standard deviation of 1.

The scaler is fitted only on training data and then applied to testing data to prevent data leakage.

In [32]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print(X_train[numeric_features].head())

           Age  Year_of_Study  Sleep_Hours  Study_Hours_Per_Week       GPA  \
726  -0.063706      -0.471636    -1.404182             -0.238603  0.074214   
1384  0.875744      -1.232340    -1.676036             -1.330997 -0.976093   
1579 -1.003157      -0.471636     0.408177             -0.516037 -1.034443   
1098  0.875744      -1.232340     1.404975             -0.082547  0.891120   
582   0.406019       0.289067     0.408177              0.663055 -0.275888   

      Social_Media_Hours  Screen_Time_Hours  Financial_Stress  Family_Support  \
726             0.148597           1.098992          0.713748        0.643008   
1384            0.327697           0.194658         -0.842691        0.368687   
1579           -0.299153          -0.830254         -0.305988        0.752736   
1098            0.775447           1.098992          0.821088       -0.948052   
582            -0.388703          -0.347943          0.660077        0.697872   

      Attendance_Rate  
726         -1.24465

In [33]:
# Check missing values
print("Train missing:", X_train.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

# Check dimensions
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Check that both datasets have the same columns
print("Same columns:", X_train.columns.equals(X_test.columns))

# Check that features and targets are aligned
print("Train indexes aligned:", X_train.index.equals(y_train.index))
print("Test indexes aligned:", X_test.index.equals(y_test.index))

Train missing: 0
Test missing: 0
Train shape: (1600, 23)
Test shape: (400, 23)
Same columns: True
Train indexes aligned: True
Test indexes aligned: True


### Check  StandarScaler

In [34]:
X_train[numeric_features].describe().round(2)

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate
count,1600.00,1600.00,1600.00,1600.00,1600.00,1600.00,1600.00,1600.00,1600.00,1600.00
mean,-0.00,0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.94,-1.23,-2.85,-3.05,-3.11,-3.52,-3.18,-2.45,-2.65,-4.26
25%,-0.53,-1.23,-0.68,-0.69,-0.68,-0.66,-0.71,-0.74,-0.73,-0.66
50%,-0.06,-0.47,-0.04,-0.03,-0.01,-0.03,-0.05,0.07,0.09,0.01
75%,0.88,1.05,0.68,0.70,0.66,0.60,0.68,0.77,0.81,0.67
max,3.22,1.81,2.95,3.18,3.20,3.55,3.27,2.16,1.85,2.08


### Verify distribuution of target

In [35]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training target distribution:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Testing target distribution:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


### Preprocessing Verification

- No missing values remain in training or testing data.
- Both datasets contain 23 features.
- Training and testing columns are identical and correctly aligned with their targets.
- Numerical features were successfully standardized using StandardScaler.
- Target classes remain balanced across both datasets.

The datasets are ready for the next preprocessing stage.